# 📐 Bài Thực Hành 4: So Sánh Sự Tương Đồng Của Các Hình Ảnh Sử Dụng Wavelet

---

**Mục tiêu bài tập:**
- Biết cách sử dụng wavelet biến đổi để trích xuất thông tin cụ thể và so sánh sự tương thích giữa các hình ảnh.
- Làm quen với PyWavelets thư viện và các công cụ xử lý ảnh trong Python.
- Đánh giá kết quả của hàm băm wavelet phương pháp trong việc xác định các hình ảnh tương thích.

**Pipeline:**
```
Đọc ảnh → Tiền xử lý (Grayscale + Resize) → DWT (PyWavelets) → Lượng tử hóa → Hash nhị phân → So sánh Hamming → Đánh giá
```


## 📦 Import Thư Viện & Cấu Hình

In [ ]:
# ============================================================
# IMPORT THƯ VIỆN
# ============================================================
import pywt
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.metrics import (
    accuracy_score, recall_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc
)
import os
import glob
from itertools import combinations

# ============================================================
# CẤU HÌNH CHUNG
# ============================================================
IMG_SIZE = (256, 256)       # Kích thước chuẩn hóa ảnh
WAVELET = 'haar'            # Loại wavelet mặc định
DWT_LEVEL = 2               # Số mức phân tích DWT
DATA_DIR = '../data/input'  # Thư mục chứa ảnh
SIMILAR_DIR = os.path.join(DATA_DIR, 'similar')
DISSIMILAR_DIR = os.path.join(DATA_DIR, 'dissimilar')
OUTPUT_DIR = '../data/input/output'

# Tạo thư mục output nếu chưa có
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SIMILAR_DIR, exist_ok=True)
os.makedirs(DISSIMILAR_DIR, exist_ok=True)

# Cấu hình matplotlib
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("=" * 60)
print("✅ Import thư viện thành công!")
print(f"   PyWavelets version : {pywt.__version__}")
print(f"   OpenCV version     : {cv2.__version__}")
print(f"   NumPy version      : {np.__version__}")
print(f"   Wavelet mặc định   : {WAVELET}")
print(f"   DWT Level          : {DWT_LEVEL}")
print(f"   Kích thước ảnh     : {IMG_SIZE}")
print("=" * 60)


---
## 🔹 Bước 1: Chuẩn Bị Dữ Liệu

> **Yêu cầu đề bài:** Chuẩn bị một tập hợp các hình ảnh, bao gồm các hình ảnh cặp đối tượng tương tự nhau (ví dụ: cùng một đối tượng với các góc độ khác nhau, các mức độ nhiễu khác nhau) và các hình ảnh cặp không tương tự.

### Cách thêm ảnh của bạn:
1. Đặt ảnh **tương tự** vào: `data/input/similar/`
2. Đặt ảnh **không tương tự** vào: `data/input/dissimilar/`

Nếu chưa có ảnh, notebook sẽ tự tạo ảnh mẫu (synthetic) để demo.


In [ ]:
# ============================================================
# BƯỚC 1.1: TẠO ẢNH MẪU (SYNTHETIC) NẾU CHƯA CÓ ẢNH THẬT
# ============================================================

def generate_synthetic_images():
    """
    Tạo bộ ảnh mẫu (synthetic) để demo nếu người dùng chưa thêm ảnh thật.
    
    Ảnh tương tự: cùng hình dạng, khác góc/nhiễu/sáng
    Ảnh không tương tự: hình dạng hoàn toàn khác nhau
    """
    print("🎨 Đang tạo ảnh mẫu (synthetic) để demo...")
    
    # --- Ảnh tương tự: Hình chữ nhật với biến thể ---
    # Ảnh gốc: hình chữ nhật trắng trên nền đen
    base_rect = np.zeros((256, 256), dtype=np.uint8)
    cv2.rectangle(base_rect, (60, 60), (200, 200), 255, -1)
    cv2.imwrite(os.path.join(SIMILAR_DIR, 'rect_original.png'), base_rect)
    
    # Biến thể 1: Xoay nhẹ
    M = cv2.getRotationMatrix2D((128, 128), 15, 1.0)
    rect_rotated = cv2.warpAffine(base_rect, M, (256, 256))
    cv2.imwrite(os.path.join(SIMILAR_DIR, 'rect_rotated.png'), rect_rotated)
    
    # Biến thể 2: Thêm nhiễu Gaussian
    noise = np.random.normal(0, 25, base_rect.shape).astype(np.int16)
    rect_noisy = np.clip(base_rect.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    cv2.imwrite(os.path.join(SIMILAR_DIR, 'rect_noisy.png'), rect_noisy)
    
    # Biến thể 3: Thay đổi độ sáng
    rect_bright = np.clip(base_rect.astype(np.int16) + 50, 0, 255).astype(np.uint8)
    cv2.imwrite(os.path.join(SIMILAR_DIR, 'rect_bright.png'), rect_bright)
    
    # Biến thể 4: Thay đổi kích thước (scale)
    rect_scaled = np.zeros((256, 256), dtype=np.uint8)
    cv2.rectangle(rect_scaled, (40, 40), (220, 220), 255, -1)
    cv2.imwrite(os.path.join(SIMILAR_DIR, 'rect_scaled.png'), rect_scaled)
    
    # --- Ảnh tương tự: Hình tròn với biến thể ---
    base_circle = np.zeros((256, 256), dtype=np.uint8)
    cv2.circle(base_circle, (128, 128), 80, 255, -1)
    cv2.imwrite(os.path.join(SIMILAR_DIR, 'circle_original.png'), base_circle)
    
    circle_noisy = np.clip(base_circle.astype(np.int16) + 
                           np.random.normal(0, 20, base_circle.shape).astype(np.int16), 
                           0, 255).astype(np.uint8)
    cv2.imwrite(os.path.join(SIMILAR_DIR, 'circle_noisy.png'), circle_noisy)
    
    circle_moved = np.zeros((256, 256), dtype=np.uint8)
    cv2.circle(circle_moved, (140, 120), 80, 255, -1)
    cv2.imwrite(os.path.join(SIMILAR_DIR, 'circle_moved.png'), circle_moved)
    
    # --- Ảnh không tương tự ---
    # Tam giác
    triangle = np.zeros((256, 256), dtype=np.uint8)
    pts = np.array([[128, 30], [30, 220], [226, 220]], np.int32)
    cv2.fillPoly(triangle, [pts], 255)
    cv2.imwrite(os.path.join(DISSIMILAR_DIR, 'triangle.png'), triangle)
    
    # Ngôi sao
    star = np.zeros((256, 256), dtype=np.uint8)
    angles = np.linspace(0, 2*np.pi, 11)[:-1]
    radii = [90, 40] * 5
    star_pts = []
    for angle, r in zip(angles, radii):
        x = int(128 + r * np.cos(angle - np.pi/2))
        y = int(128 + r * np.sin(angle - np.pi/2))
        star_pts.append([x, y])
    cv2.fillPoly(star, [np.array(star_pts, np.int32)], 255)
    cv2.imwrite(os.path.join(DISSIMILAR_DIR, 'star.png'), star)
    
    # Gradient ngang
    gradient_h = np.tile(np.linspace(0, 255, 256, dtype=np.uint8), (256, 1))
    cv2.imwrite(os.path.join(DISSIMILAR_DIR, 'gradient_h.png'), gradient_h)
    
    # Gradient dọc
    gradient_v = np.tile(np.linspace(0, 255, 256, dtype=np.uint8).reshape(-1,1), (1, 256))
    cv2.imwrite(os.path.join(DISSIMILAR_DIR, 'gradient_v.png'), gradient_v)
    
    # Checkerboard
    checker = np.zeros((256, 256), dtype=np.uint8)
    for i in range(0, 256, 32):
        for j in range(0, 256, 32):
            if (i // 32 + j // 32) % 2 == 0:
                checker[i:i+32, j:j+32] = 255
    cv2.imwrite(os.path.join(DISSIMILAR_DIR, 'checkerboard.png'), checker)
    
    # Đường chéo
    diag = np.zeros((256, 256), dtype=np.uint8)
    for i in range(0, 256, 16):
        cv2.line(diag, (i, 0), (i+128, 256), 255, 3)
    cv2.imwrite(os.path.join(DISSIMILAR_DIR, 'diagonal_lines.png'), diag)
    
    print("✅ Đã tạo ảnh mẫu thành công!")
    print(f"   Ảnh tương tự (similar)      : {SIMILAR_DIR}")
    print(f"   Ảnh không tương tự (dissimilar): {DISSIMILAR_DIR}")


# Kiểm tra xem đã có ảnh thật chưa
similar_imgs = glob.glob(os.path.join(SIMILAR_DIR, '*.*'))
dissimilar_imgs = glob.glob(os.path.join(DISSIMILAR_DIR, '*.*'))

if len(similar_imgs) < 2 or len(dissimilar_imgs) < 2:
    print("⚠️  Chưa có đủ ảnh trong thư mục data/input/")
    print("    → Tạo ảnh mẫu (synthetic) để demo...\n")
    generate_synthetic_images()
else:
    print(f"✅ Đã tìm thấy ảnh thật!")
    print(f"   Ảnh tương tự     : {len(similar_imgs)} ảnh")
    print(f"   Ảnh không tương tự: {len(dissimilar_imgs)} ảnh")


In [ ]:
# ============================================================
# BƯỚC 1.2: TẢI VÀ HIỂN THỊ ẢNH
# ============================================================

def load_and_preprocess(image_path: str, size: tuple = IMG_SIZE) -> np.ndarray:
    """
    Đọc ảnh từ file, chuyển sang grayscale và resize.
    
    Args:
        image_path: Đường dẫn file ảnh
        size: Kích thước đầu ra (width, height)
    
    Returns:
        img: Ảnh grayscale đã resize (numpy array 2D)
    
    Raises:
        FileNotFoundError: Nếu không tìm thấy file ảnh
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Không tìm thấy ảnh: {image_path}")
    img = cv2.resize(img, size)
    return img


# Tải tất cả ảnh
similar_paths = sorted(glob.glob(os.path.join(SIMILAR_DIR, '*.*')))
dissimilar_paths = sorted(glob.glob(os.path.join(DISSIMILAR_DIR, '*.*')))
all_paths = similar_paths + dissimilar_paths

print(f"📂 Tổng số ảnh: {len(all_paths)}")
print(f"   Tương tự     : {len(similar_paths)} ảnh")
print(f"   Không tương tự: {len(dissimilar_paths)} ảnh")
print()

# Tải ảnh vào dict {tên_file: numpy_array}
images = {}
for path in all_paths:
    name = os.path.basename(path)
    images[name] = load_and_preprocess(path)
    print(f"   ✅ {name}: {images[name].shape}")

print(f"\n📊 Đã tải thành công {len(images)} ảnh.")


In [ ]:
# ============================================================
# BƯỚC 1.3: HIỂN THỊ ẢNH ĐÃ TẢI
# ============================================================

def display_images(images_dict: dict, title: str = "Tập Ảnh Đã Tải", max_per_row: int = 5):
    """Hiển thị tất cả ảnh đã tải dưới dạng grid."""
    n = len(images_dict)
    rows = (n + max_per_row - 1) // max_per_row
    
    fig, axes = plt.subplots(rows, max_per_row, figsize=(3*max_per_row, 3.5*rows))
    if rows == 1:
        axes = axes.reshape(1, -1) if n > 1 else np.array([[axes]])
    
    for idx, (name, img) in enumerate(images_dict.items()):
        r, c = idx // max_per_row, idx % max_per_row
        axes[r, c].imshow(img, cmap='gray')
        axes[r, c].set_title(name, fontsize=9)
        axes[r, c].axis('off')
    
    # Ẩn axes thừa
    for idx in range(n, rows * max_per_row):
        r, c = idx // max_per_row, idx % max_per_row
        axes[r, c].axis('off')
    
    fig.suptitle(title, fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Hiển thị ảnh tương tự
similar_images = {k: v for k, v in images.items() 
                  if os.path.join(SIMILAR_DIR, k) in similar_paths 
                  or any(k in p for p in similar_paths)}
dissimilar_images = {k: v for k, v in images.items() 
                     if os.path.join(DISSIMILAR_DIR, k) in dissimilar_paths
                     or any(k in p for p in dissimilar_paths)}

print("🟢 ẢNH TƯƠNG TỰ:")
display_images(similar_images, "🟢 Ảnh Tương Tự (Similar)")

print("\n🔴 ẢNH KHÔNG TƯƠNG TỰ:")
display_images(dissimilar_images, "🔴 Ảnh Không Tương Tự (Dissimilar)")


---
## 🔹 Bước 2: Trích Xuất Wavelet Đặc Biệt

> **Yêu cầu đề bài:** Sử dụng wavelet biến đổi để chuyển đổi từng hình ảnh thành một wavelet ma trận.

### Giải thích:
Biến đổi Wavelet rời rạc 2D (2D DWT) phân tích ảnh thành 4 thành phần (sub-bands):

| Sub-band | Tên | Ý nghĩa |
|----------|-----|---------|
| **cA** | Approximation | Thông tin tổng thể (tần số thấp) |
| **cH** | Horizontal Detail | Các cạnh ngang |
| **cV** | Vertical Detail | Các cạnh dọc |
| **cD** | Diagonal Detail | Các cạnh chéo |


In [ ]:
# ============================================================
# BƯỚC 2.1: HÀM TRÍCH XUẤT WAVELET
# ============================================================

def extract_wavelet(img: np.ndarray, wavelet: str = WAVELET, level: int = DWT_LEVEL):
    """
    Áp dụng biến đổi wavelet rời rạc 2D (DWT) cho ảnh.
    
    Sử dụng hàm pywt.wavedec2() để phân tích ảnh thành các hệ số wavelet
    ở nhiều mức (multi-level decomposition).
    
    Args:
        img: Ảnh grayscale (numpy array 2D)
        wavelet: Loại wavelet ('haar', 'db1', 'db2', 'sym2', ...)
        level: Số mức phân tích DWT
    
    Returns:
        coeffs: Danh sách hệ số wavelet
                [cA_n, (cH_n, cV_n, cD_n), ..., (cH_1, cV_1, cD_1)]
    """
    coeffs = pywt.wavedec2(img.astype(np.float64), wavelet, level=level)
    return coeffs


# Trích xuất wavelet cho tất cả ảnh
wavelet_coeffs = {}
for name, img in images.items():
    wavelet_coeffs[name] = extract_wavelet(img)

print(f"✅ Đã trích xuất wavelet cho {len(wavelet_coeffs)} ảnh")
print(f"   Wavelet: {WAVELET}, Level: {DWT_LEVEL}")
print()

# Hiển thị thông tin cấu trúc wavelet cho 1 ảnh mẫu
sample_name = list(wavelet_coeffs.keys())[0]
sample_coeffs = wavelet_coeffs[sample_name]
print(f"📐 Cấu trúc hệ số wavelet của '{sample_name}':")
print(f"   Level 0 (cA) shape: {sample_coeffs[0].shape}")
for i in range(1, len(sample_coeffs)):
    cH, cV, cD = sample_coeffs[i]
    print(f"   Level {i} (cH, cV, cD) shape: {cH.shape}")


In [ ]:
# ============================================================
# BƯỚC 2.2: TRỰC QUAN HÓA WAVELET DECOMPOSITION
# ============================================================

def visualize_wavelet(img: np.ndarray, coeffs, title: str = ""):
    """
    Hiển thị ảnh gốc và 4 sub-bands wavelet (cA, cH, cV, cD).
    
    Args:
        img: Ảnh gốc (grayscale)
        coeffs: Hệ số wavelet từ hàm extract_wavelet()
        title: Tiêu đề
    """
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    # Ảnh gốc
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title('Ảnh gốc', fontweight='bold')
    axes[0].axis('off')
    
    # Sub-bands
    cA = coeffs[0]
    cH, cV, cD = coeffs[1]
    
    sub_bands = [
        (cA, 'cA (Xấp xỉ)', 'gray'),
        (cH, 'cH (Ngang)', 'RdBu'),
        (cV, 'cV (Dọc)', 'RdBu'),
        (cD, 'cD (Chéo)', 'RdBu')
    ]
    
    for i, (band, name, cmap) in enumerate(sub_bands):
        axes[i+1].imshow(np.abs(band), cmap=cmap)
        axes[i+1].set_title(name, fontweight='bold')
        axes[i+1].axis('off')
    
    if title:
        fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# Trực quan hóa wavelet cho một số ảnh mẫu
sample_names = list(images.keys())[:4]  # Lấy 4 ảnh đầu tiên
for name in sample_names:
    visualize_wavelet(images[name], wavelet_coeffs[name], 
                      title=f"🌊 Wavelet Decomposition: {name}")
    print()


---
## 🔹 Bước 3: Tạo Mã Băm (Wavelet Hash)

> **Yêu cầu đề bài:** Tạo mã băm cho mỗi hình ảnh dựa trên các wavelet số đã lượng tử hóa.

### Giải thích phương pháp:
1. Lấy hệ số xấp xỉ `cA` (approximation coefficients) từ DWT
2. Tính giá trị **trung bình** (mean) của hệ số cA
3. **Lượng tử hóa**: Nếu hệ số ≥ mean → `1`, ngược lại → `0`
4. Kết quả: Một chuỗi nhị phân (binary hash) đại diện cho ảnh

> 💡 Ảnh tương tự sẽ cho ra hash tương tự nhau (ít bit khác biệt).


In [ ]:
# ============================================================
# BƯỚC 3.1: HÀM TẠO MÃ BĂM WAVELET
# ============================================================

def create_wavelet_hash(coeffs) -> np.ndarray:
    """
    Tạo mã băm nhị phân từ hệ số wavelet.
    
    Phương pháp lượng tử hóa:
    - Lấy hệ số xấp xỉ (cA) từ mức cao nhất
    - So sánh mỗi hệ số với giá trị trung bình
    - Hệ số >= mean → bit 1
    - Hệ số <  mean → bit 0
    
    Args:
        coeffs: Hệ số wavelet từ hàm extract_wavelet()
    
    Returns:
        binary_hash: Mảng nhị phân 1D (numpy array, dtype=uint8)
    """
    # Lấy hệ số xấp xỉ (cA) - chứa thông tin tổng thể của ảnh
    approx = coeffs[0]
    
    # Tính ngưỡng: giá trị trung bình
    mean_val = np.mean(approx)
    
    # Lượng tử hóa: >= mean → 1, < mean → 0
    binary_hash = (approx >= mean_val).astype(np.uint8).flatten()
    
    return binary_hash


def hash_to_string(binary_hash: np.ndarray) -> str:
    """Chuyển mảng nhị phân thành chuỗi string dễ đọc."""
    return ''.join(map(str, binary_hash))


# Tạo hash cho tất cả ảnh
hashes = {}
for name, coeffs in wavelet_coeffs.items():
    hashes[name] = create_wavelet_hash(coeffs)

# Hiển thị kết quả
print("🔑 MÃ BĂM WAVELET CỦA CÁC ẢNH:")
print("=" * 70)
for name, h in hashes.items():
    hash_str = hash_to_string(h)
    preview = hash_str[:50] + "..." if len(hash_str) > 50 else hash_str
    print(f"  {name:30s} | Kích thước: {len(h):5d} bit | Hash: {preview}")
print("=" * 70)
print(f"\n📊 Tổng: {len(hashes)} mã băm đã tạo, mỗi hash có {len(list(hashes.values())[0])} bit")


---
## 🔹 Bước 4: So Sánh Hàm Băm (Hamming Distance)

> **Yêu cầu đề bài:** Tính khoảng cách Hamming giữa các mã băm để đánh giá mức độ tương thích giữa các hình ảnh.

### Giải thích:
**Khoảng cách Hamming** = số vị trí mà hai chuỗi nhị phân khác nhau.

| Hamming Distance | Ý nghĩa |
|:---:|:---|
| **Nhỏ** (gần 0) | Hai ảnh rất **tương tự** |
| **Lớn** (gần max) | Hai ảnh rất **khác biệt** |


In [ ]:
# ============================================================
# BƯỚC 4.1: HÀM TÍNH KHOẢNG CÁCH HAMMING
# ============================================================

def hamming_distance(hash1: np.ndarray, hash2: np.ndarray) -> tuple:
    """
    Tính khoảng cách Hamming giữa 2 mã băm.
    
    Khoảng cách Hamming = số bit khác nhau giữa 2 chuỗi nhị phân.
    
    Args:
        hash1, hash2: Mảng nhị phân cùng kích thước
    
    Returns:
        (distance, similarity):
            distance   - Số bit khác nhau (int)
            similarity - Tỷ lệ tương đồng (float, 0.0 → 1.0)
    """
    if len(hash1) != len(hash2):
        raise ValueError(f"Hash phải cùng kích thước! ({len(hash1)} vs {len(hash2)})")
    
    # Đếm số bit khác nhau
    distance = int(np.sum(hash1 != hash2))
    
    # Tỷ lệ tương đồng = 1 - (số bit khác / tổng số bit)
    similarity = 1.0 - (distance / len(hash1))
    
    return distance, similarity


In [ ]:
# ============================================================
# BƯỚC 4.2: ĐỊNH NGHĨA CÁC CẶP ẢNH VÀ NHÃN
# ============================================================

# Tạo danh sách tên ảnh theo nhóm
similar_names = [os.path.basename(p) for p in similar_paths]
dissimilar_names = [os.path.basename(p) for p in dissimilar_paths]

# Tạo các cặp ảnh với nhãn
#   label = 1 → cặp tương tự (cùng nhóm similar)
#   label = 0 → cặp không tương tự (1 similar + 1 dissimilar)
pairs = []
labels = []

# Cặp tương tự: tổ hợp 2 ảnh từ thư mục similar
for img1, img2 in combinations(similar_names, 2):
    pairs.append((img1, img2))
    labels.append(1)

# Cặp không tương tự: 1 ảnh similar + 1 ảnh dissimilar
for s_img in similar_names:
    for d_img in dissimilar_names:
        pairs.append((s_img, d_img))
        labels.append(0)

# Thêm cặp dissimilar với nhau (cũng là không tương tự)
for img1, img2 in combinations(dissimilar_names, 2):
    pairs.append((img1, img2))
    labels.append(0)

print(f"📊 Tổng số cặp ảnh: {len(pairs)}")
print(f"   Cặp tương tự (label=1)     : {labels.count(1)}")
print(f"   Cặp không tương tự (label=0): {labels.count(0)}")


In [ ]:
# ============================================================
# BƯỚC 4.3: TÍNH KHOẢNG CÁCH HAMMING CHO TẤT CẢ CÁC CẶP
# ============================================================

distances = []
similarities = []

print("📏 BẢNG KHOẢNG CÁCH HAMMING:")
print("=" * 85)
print(f"  {'Ảnh 1':25s} | {'Ảnh 2':25s} | {'Distance':>8s} | {'Similarity':>10s} | {'Label':>5s}")
print("-" * 85)

for (img1, img2), label in zip(pairs, labels):
    dist, sim = hamming_distance(hashes[img1], hashes[img2])
    distances.append(dist)
    similarities.append(sim)
    
    label_icon = "🟢" if label == 1 else "🔴"
    print(f"  {img1:25s} | {img2:25s} | {dist:8d} | {sim:10.4f} | {label_icon} {label}")

print("=" * 85)
print(f"\n📊 Thống kê:")
print(f"   Similarity trung bình (cặp tương tự)     : {np.mean([s for s, l in zip(similarities, labels) if l == 1]):.4f}")
print(f"   Similarity trung bình (cặp không tương tự): {np.mean([s for s, l in zip(similarities, labels) if l == 0]):.4f}")


In [ ]:
# ============================================================
# BƯỚC 4.4: TRỰC QUAN HÓA PHÂN BỐ KHOẢNG CÁCH
# ============================================================

sim_similar = [s for s, l in zip(similarities, labels) if l == 1]
sim_dissimilar = [s for s, l in zip(similarities, labels) if l == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(sim_similar, bins=15, alpha=0.7, color='#2ecc71', label='Tương tự (label=1)', edgecolor='white')
axes[0].hist(sim_dissimilar, bins=15, alpha=0.7, color='#e74c3c', label='Không tương tự (label=0)', edgecolor='white')
axes[0].set_xlabel('Similarity Score', fontsize=12)
axes[0].set_ylabel('Số cặp ảnh', fontsize=12)
axes[0].set_title('📊 Phân Bố Similarity Score', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Box plot
bp = axes[1].boxplot([sim_similar, sim_dissimilar], 
                     labels=['Tương tự', 'Không tương tự'],
                     patch_artist=True)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][1].set_facecolor('#e74c3c')
for box in bp['boxes']:
    box.set_alpha(0.7)
axes[1].set_ylabel('Similarity Score', fontsize=12)
axes[1].set_title('📦 Box Plot So Sánh', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Nhận xét: Nếu 2 nhóm tách biệt rõ ràng → thuật toán phân loại tốt.")


---
## 🔹 Bước 5: Đánh Giá

> **Yêu cầu đề bài:** Tính các metrics đánh giá:
> - **Độ chính xác (Accuracy):** Tỷ lệ tính toán của các cặp hình ảnh được phân loại đúng.
> - **Độ nhạy (Sensitivity/Recall):** Tỷ lệ các cặp hình ảnh tương thích được phân loại đúng.
> - **Độ đặc biệt (Specificity):** Tỷ lệ tính toán của các cặp hình ảnh không tương thích với loại phân tích đúng.
> - **Đường cong ROC:** Vẽ đường cong ROC để đánh giá hiệu suất của thuật toán.


In [ ]:
# ============================================================
# BƯỚC 5.1: CHỌN NGƯỠNG VÀ TÍNH CÁC METRICS
# ============================================================

def evaluate_with_threshold(y_true, y_scores, threshold):
    """
    Đánh giá kết quả phân loại với ngưỡng (threshold) cho trước.
    
    - Nếu similarity >= threshold → dự đoán "tương tự" (1)
    - Nếu similarity <  threshold → dự đoán "không tương tự" (0)
    
    Args:
        y_true: Nhãn thực tế (list of 0/1)
        y_scores: Similarity scores (list of float)
        threshold: Ngưỡng phân loại (float)
    
    Returns:
        dict chứa các metrics
    """
    # Phân loại dựa trên ngưỡng
    y_pred = [1 if s >= threshold else 0 for s in y_scores]
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    # Tính metrics
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # Recall
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    return {
        'threshold': threshold,
        'accuracy': accuracy,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'confusion_matrix': cm,
        'y_pred': y_pred
    }


# Thử nhiều ngưỡng để tìm ngưỡng tối ưu
thresholds = np.arange(0.5, 1.0, 0.05)
results = []

print("📊 ĐÁNH GIÁ VỚI CÁC NGƯỠNG KHÁC NHAU:")
print("=" * 90)
print(f"  {'Threshold':>9s} | {'Accuracy':>8s} | {'Sensitivity':>11s} | {'Specificity':>11s} | {'TP':>3s} | {'TN':>3s} | {'FP':>3s} | {'FN':>3s}")
print("-" * 90)

for t in thresholds:
    result = evaluate_with_threshold(labels, similarities, t)
    results.append(result)
    print(f"  {t:9.2f} | {result['accuracy']:8.4f} | {result['sensitivity']:11.4f} | {result['specificity']:11.4f} | {result['tp']:3d} | {result['tn']:3d} | {result['fp']:3d} | {result['fn']:3d}")

print("=" * 90)

# Tìm ngưỡng tốt nhất (accuracy cao nhất)
best_result = max(results, key=lambda x: x['accuracy'])
best_threshold = best_result['threshold']

print(f"\n🏆 NGƯỠNG TỐI ƯU: {best_threshold:.2f}")
print(f"   Accuracy    : {best_result['accuracy']:.4f} ({best_result['accuracy']*100:.2f}%)")
print(f"   Sensitivity : {best_result['sensitivity']:.4f} ({best_result['sensitivity']*100:.2f}%)")
print(f"   Specificity : {best_result['specificity']:.4f} ({best_result['specificity']*100:.2f}%)")


In [ ]:
# ============================================================
# BƯỚC 5.2: CONFUSION MATRIX TRỰC QUAN
# ============================================================

# Sử dụng ngưỡng tối ưu
y_pred_best = best_result['y_pred']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = best_result['confusion_matrix']
disp = ConfusionMatrixDisplay(cm, display_labels=['Không tương tự', 'Tương tự'])
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title(f'📊 Confusion Matrix (Threshold = {best_threshold:.2f})', 
                  fontsize=13, fontweight='bold')

# Metrics bar chart
metrics_names = ['Accuracy', 'Sensitivity', 'Specificity']
metrics_values = [best_result['accuracy'], best_result['sensitivity'], best_result['specificity']]
colors = ['#3498db', '#2ecc71', '#e67e22']

bars = axes[1].bar(metrics_names, metrics_values, color=colors, alpha=0.8, edgecolor='white', linewidth=2)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Giá trị', fontsize=12)
axes[1].set_title('📈 Các Metrics Đánh Giá', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Thêm giá trị lên bar
for bar, val in zip(bars, metrics_values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{val:.2%}', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# BƯỚC 5.3: ĐƯỜNG CONG ROC (ROC CURVE)
# ============================================================

# Tính ROC curve
fpr, tpr, roc_thresholds = roc_curve(labels, similarities)
roc_auc = auc(fpr, tpr)

# Tìm điểm tối ưu trên ROC (Youden's J statistic)
j_scores = tpr - fpr
best_j_idx = np.argmax(j_scores)
best_roc_threshold = roc_thresholds[best_j_idx]

# Vẽ ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC Curve
axes[0].plot(fpr, tpr, color='#00BFFF', lw=2.5, 
             label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', 
             label='Random Classifier (AUC = 0.5)', alpha=0.7)
axes[0].scatter(fpr[best_j_idx], tpr[best_j_idx], color='red', s=100, zorder=5,
                label=f'Best Point (threshold={best_roc_threshold:.3f})')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#00BFFF')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
axes[0].set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
axes[0].set_title('📈 Đường Cong ROC', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Threshold vs Metrics
eval_thresholds = np.linspace(0.3, 1.0, 50)
accs, sens, specs = [], [], []
for t in eval_thresholds:
    r = evaluate_with_threshold(labels, similarities, t)
    accs.append(r['accuracy'])
    sens.append(r['sensitivity'])
    specs.append(r['specificity'])

axes[1].plot(eval_thresholds, accs, label='Accuracy', lw=2, color='#3498db')
axes[1].plot(eval_thresholds, sens, label='Sensitivity', lw=2, color='#2ecc71')
axes[1].plot(eval_thresholds, specs, label='Specificity', lw=2, color='#e67e22')
axes[1].axvline(x=best_threshold, color='red', linestyle='--', alpha=0.7, 
                label=f'Best Threshold = {best_threshold:.2f}')
axes[1].set_xlabel('Threshold', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('📊 Metrics vs Threshold', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📌 KẾT QUẢ ROC:")
print(f"   AUC (Area Under Curve)        : {roc_auc:.4f}")
print(f"   Ngưỡng tối ưu (Youden's J)    : {best_roc_threshold:.4f}")
print(f"   True Positive Rate tại điểm đó: {tpr[best_j_idx]:.4f}")
print(f"   False Positive Rate tại điểm đó: {fpr[best_j_idx]:.4f}")


---
## 🔹 Phần Nâng Cao: Khảo Sát Các Phương Pháp Băm Wavelet Khác Nhau

> **Yêu cầu đề bài (III - Bài tập nâng cao):**  
> 1. Thực hiện khảo sát về các phương pháp băm wavelet khác nhau và so sánh hiệu suất của chúng.

### Các loại wavelet sẽ thử nghiệm:

| Wavelet | Tên đầy đủ | Đặc điểm |
|---------|-----------|-----------|
| `haar` | Haar | Đơn giản nhất, nhanh, phù hợp ảnh có cạnh sắc |
| `db2` | Daubechies-2 | Mượt hơn Haar, phổ biến |
| `db4` | Daubechies-4 | Bộ lọc dài hơn, chi tiết hơn |
| `sym2` | Symlet-2 | Đối xứng gần, ít artifact |
| `coif1` | Coiflet-1 | Cân bằng giữa độ mượt và chi tiết |
| `bior1.3` | Biorthogonal 1.3 | Tái tạo hoàn hảo, linh hoạt |


In [ ]:
# ============================================================
# NÂNG CAO: SO SÁNH CÁC PHƯƠNG PHÁP WAVELET KHÁC NHAU
# ============================================================

wavelet_types = ['haar', 'db2', 'db4', 'sym2', 'coif1', 'bior1.3']

comparison_results = []

print("🔬 SO SÁNH CÁC PHƯƠNG PHÁP WAVELET:")
print("=" * 95)
print(f"  {'Wavelet':>10s} | {'AUC':>8s} | {'Best Threshold':>14s} | {'Accuracy':>8s} | {'Sensitivity':>11s} | {'Specificity':>11s}")
print("-" * 95)

for wv in wavelet_types:
    try:
        # Trích xuất wavelet với loại wavelet khác nhau
        wv_hashes = {}
        for name, img in images.items():
            coeffs = extract_wavelet(img, wavelet=wv, level=DWT_LEVEL)
            wv_hashes[name] = create_wavelet_hash(coeffs)
        
        # Tính similarity cho tất cả cặp
        wv_similarities = []
        for img1, img2 in pairs:
            _, sim = hamming_distance(wv_hashes[img1], wv_hashes[img2])
            wv_similarities.append(sim)
        
        # Tính ROC AUC
        fpr_wv, tpr_wv, thresh_wv = roc_curve(labels, wv_similarities)
        auc_wv = auc(fpr_wv, tpr_wv)
        
        # Tìm ngưỡng tối ưu
        j_wv = tpr_wv - fpr_wv
        best_idx = np.argmax(j_wv)
        best_t = thresh_wv[best_idx]
        
        # Đánh giá với ngưỡng tối ưu
        eval_result = evaluate_with_threshold(labels, wv_similarities, best_t)
        
        comparison_results.append({
            'wavelet': wv,
            'auc': auc_wv,
            'best_threshold': best_t,
            'accuracy': eval_result['accuracy'],
            'sensitivity': eval_result['sensitivity'],
            'specificity': eval_result['specificity'],
            'fpr': fpr_wv,
            'tpr': tpr_wv
        })
        
        print(f"  {wv:>10s} | {auc_wv:8.4f} | {best_t:14.4f} | {eval_result['accuracy']:8.4f} | {eval_result['sensitivity']:11.4f} | {eval_result['specificity']:11.4f}")
    
    except Exception as e:
        print(f"  {wv:>10s} | ❌ Lỗi: {e}")

print("=" * 95)

# Tìm wavelet tốt nhất
best_wv = max(comparison_results, key=lambda x: x['auc'])
print(f"\n🏆 WAVELET TỐT NHẤT: {best_wv['wavelet']} (AUC = {best_wv['auc']:.4f})")


In [ ]:
# ============================================================
# NÂNG CAO: TRỰC QUAN HÓA SO SÁNH ROC CỦA CÁC WAVELET
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curves cho tất cả wavelet
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
for i, result in enumerate(comparison_results):
    axes[0].plot(result['fpr'], result['tpr'], 
                 color=colors[i % len(colors)], lw=2,
                 label=f"{result['wavelet']} (AUC={result['auc']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('📈 So Sánh ROC Curves - Các Loại Wavelet', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9, loc='lower right')
axes[0].grid(True, alpha=0.3)

# Bar chart AUC
wv_names = [r['wavelet'] for r in comparison_results]
auc_values = [r['auc'] for r in comparison_results]
bar_colors = [colors[i % len(colors)] for i in range(len(comparison_results))]

bars = axes[1].bar(wv_names, auc_values, color=bar_colors, alpha=0.8, edgecolor='white', linewidth=2)
axes[1].set_ylabel('AUC Score', fontsize=12)
axes[1].set_title('📊 So Sánh AUC - Các Loại Wavelet', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 1.15)
axes[1].grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, auc_values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()


---
## 📝 Kết Luận

### Tóm tắt:
1. **Biến đổi Wavelet (DWT)** phân tích ảnh thành các sub-band tần số khác nhau (cA, cH, cV, cD), cho phép trích xuất đặc trưng ảnh hiệu quả.

2. **Wavelet Hashing** tạo "dấu vân tay" (fingerprint) nhị phân cho mỗi ảnh bằng cách lượng tử hóa hệ số wavelet.

3. **Khoảng cách Hamming** là phương pháp so sánh nhanh và hiệu quả giữa các hash, cho phép phân loại ảnh tương tự / không tương tự.

4. **Đánh giá** bằng Accuracy, Sensitivity, Specificity và ROC Curve cho thấy hiệu suất của thuật toán.

5. **So sánh wavelet**: Các loại wavelet khác nhau cho kết quả khác nhau tùy thuộc vào đặc điểm của tập dữ liệu.

### Hạn chế:
- Phương pháp nhạy cảm với biến đổi hình học lớn (xoay > 30°, scale thay đổi nhiều)
- Ngưỡng Hamming cần được tinh chỉnh cho từng tập dữ liệu cụ thể

### Hướng phát triển:
- Kết hợp nhiều mức wavelet (multi-level) để tăng độ chính xác
- Sử dụng kết hợp hash từ cả cA và các sub-band chi tiết (cH, cV, cD)
- Tích hợp vào ứng dụng web tìm kiếm ảnh tương tự


In [ ]:
# ============================================================
# TỔNG KẾT - IN KẾT QUẢ CUỐI CÙNG
# ============================================================

print("=" * 60)
print("📋 TỔNG KẾT BÀI THỰC HÀNH 4")
print("=" * 60)
print(f"  Số lượng ảnh         : {len(images)}")
print(f"  Số cặp ảnh           : {len(pairs)}")
print(f"  Cặp tương tự         : {labels.count(1)}")
print(f"  Cặp không tương tự   : {labels.count(0)}")
print(f"  Kích thước hash      : {len(list(hashes.values())[0])} bit")
print(f"  Wavelet mặc định     : {WAVELET}")
print(f"  Ngưỡng tối ưu        : {best_threshold:.2f}")
print(f"  Accuracy             : {best_result['accuracy']:.2%}")
print(f"  Sensitivity (Recall) : {best_result['sensitivity']:.2%}")
print(f"  Specificity          : {best_result['specificity']:.2%}")
print(f"  AUC                  : {roc_auc:.4f}")
print(f"  Wavelet tốt nhất     : {best_wv['wavelet']} (AUC={best_wv['auc']:.4f})")
print("=" * 60)
print("\n✅ Hoàn thành bài thực hành!")
